# gas-sim-pro · Training Notebook
Run all cells top to bottom. Do not skip cells.
Training completes in ~10 minutes on Colab free tier.

In [ ]:
# ── Cell 1 — Config + registry check ─────────────────────────────────────
PROJECT_ID = 'gas-sim-pro-ii'   # ← your project ID
BUCKET     = f'{PROJECT_ID}-gas-sim-data'

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

gcs    = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)
reg    = json.loads(bucket.blob('model_registry.json').download_as_text())

print('── Registry ──────────────────────────────')
print(f'  last_trained:     {reg["last_trained"]}')
print(f'  last_data_upload: {reg["last_data_upload"]}')
print(f'  feature_version:  {reg["feature_version"]}')
print(f'  current MAE:      {reg["mae"]}')
print(f'  rows_trained_on:  {reg["rows_trained_on"]}')
print('──────────────────────────────────────────')
print('Registry loaded. Proceed to Cell 2.')

In [ ]:
# ── Cell 2 — Install dependencies ────────────────────────────────────────
!pip install -q xgboost wandb onnx onnxmltools skl2onnx pyarrow pandas-gbq

In [ ]:
# ── Cell 3 — Load features from GCS ──────────────────────────────────────
import pandas as pd

# Find all parquet files for this feature version
blobs = list(gcs.bucket(BUCKET).list_blobs(prefix='features/latest/'))
paths = [f'gs://{BUCKET}/{b.name}' for b in blobs if b.name.endswith('.parquet')]
print(f'Found {len(paths)} parquet file(s)')

df = pd.concat(
    [pd.read_parquet(p, storage_options={'token': 'google_default'}) for p in paths],
    ignore_index=True
)

assert len(df) >= 500, f'Only {len(df)} rows — need at least 500 to train'
print(f'Loaded {len(df):,} rows · {len(df.columns)} columns')
df[['sensor_delta','centroid_row','centroid_col','wind_angle','leak_row','leak_col']].describe()

In [ ]:
# ── Cell 4 — Feature matrix + train/val split ─────────────────────────────
from sklearn.model_selection import train_test_split
import numpy as np

FEATURES = [
    'sensor_delta', 'sensor_mean', 'reading_variance',
    'centroid_row', 'centroid_col', 'coverage_ratio',
    'wind_angle', 'wind_magnitude', 'distance_to_boundary',
    'wind_x', 'wind_y', 'diffusion_rate', 'decay_factor',
    'leak_injection', 'sensor_count'
]
TARGETS = ['leak_row', 'leak_col']

X = df[FEATURES].values.astype(np.float32)
y = df[TARGETS].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42
)
print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

In [ ]:
# ── Cell 5 — W&B init (optional) ────────────────────────────────────────
import wandb

try:
    run = wandb.init(
        project="gas-sim-pro",
        anonymous="allow",   # never prompts for login
        mode="offline",      # works without internet, sync later if desired
        config={
            "model": "xgboost",
            "features": FEATURES,
            "n_rows": len(df),
            "feature_version": reg["feature_version"],
        }
    )
    WANDB_ACTIVE = True
    print(f"W&B run: {run.url}")
except Exception as e:
    print(f"W&B unavailable ({e}) — continuing without experiment tracking")
    run = None
    WANDB_ACTIVE = False


In [ ]:
# ── Cell 7 — MAE regression gate ─────────────────────────────────────────
# Reload registry fresh — avoids stale values from Cell 1 load
reg = json.loads(bucket.blob("model_registry.json").download_as_text())

prod_mae = reg.get("mae") or float("inf")

if mae >= prod_mae * 0.98:
    print(f"Gate FAILED: new MAE {mae:.3f} does not beat prod {prod_mae:.3f} by >2%")
    wandb.finish(exit_code=1)
    raise SystemExit("MAE gate failed — no model exported")

print(f"Gate PASSED: {mae:.3f} < {prod_mae:.3f} * 0.98")


In [ ]:
# ── Cell 7 — MAE gate (advisory) ────────────────────────────────────────
# Re-read registry fresh — may have changed since Cell 1 (e.g. manual reset)
reg_fresh = json.loads(bucket.blob("model_registry.json").download_as_text())
prod_mae  = reg_fresh.get("mae") or float("inf")
prev_mae  = reg_fresh.get("previous_mae")

print("── MAE Gate Assessment ──────────────────────────────────────")
print(f"  New model MAE  : {mae:.4f} cells")
print(f"  Current prod   : {prod_mae:.4f} cells")
print(f"  Previous prod  : {prev_mae}")
print(f"  Gate threshold : prod × 0.98 = {prod_mae * 0.98:.4f}")
print()

if prod_mae == float("inf"):
    GATE_STATUS = "first_model"
    print("✅ GATE: First model — no baseline to beat. Will deploy.")

elif mae < prod_mae * 0.98:
    GATE_STATUS = "passed"
    improvement = (prod_mae - mae) / prod_mae * 100
    print(f"✅ GATE PASSED: {improvement:.1f}% improvement over production.")
    print("   Model will be deployed.")

elif mae < prod_mae:
    GATE_STATUS = "marginal"
    diff = (prod_mae - mae) / prod_mae * 100
    print(f"⚠️  GATE MARGINAL: {diff:.1f}% better but below the 2% threshold.")
    print("   Model is slightly better but not enough to trigger auto-deploy.")
    print("   Consider: generate more data and retrain, or lower the threshold.")
    print("   Continuing — registry will be updated with advisory gate_status.")

else:
    GATE_STATUS = "failed"
    regressed = (mae - prod_mae) / prod_mae * 100
    print(f"❌ GATE FAILED: New model is {regressed:.1f}% WORSE than production.")
    print(f"   prod MAE: {prod_mae:.4f}  →  new MAE: {mae:.4f}")
    print()
    print("   Possible causes:")
    print("   • Not enough new training data (try 500k+ rows)")
    print("   • Same dataset trained twice — generate fresh data first")
    print("   • Feature distribution changed — check dbt export")
    print()
    print("   The notebook will continue and export the model.")
    print("   The deploy function will also check the gate and may block deployment.")
    print("   To force deployment anyway: run cloud.sh → option 5 (reset registry).")

if WANDB_ACTIVE and run:
    wandb.log({"gate_status": GATE_STATUS, "prod_mae": prod_mae})

print("─────────────────────────────────────────────────────────────")


In [ ]:
# ── Cell 8 — Export model ────────────────────────────────────────────────
import os, datetime, joblib

if GATE_STATUS not in ("passed", "first_model"):
    print(f"⏭  Cell 8 skipped — gate status is [{GATE_STATUS}]")
    print("   Model will not be exported or uploaded.")
    VERSION = None
else:
    RUN_ID  = run.id if WANDB_ACTIVE and run else datetime.datetime.now().strftime("%H%M%S")
    VERSION = f"v{datetime.date.today().strftime('%Y%m%d')}-{RUN_ID}"
    os.makedirs(f"/tmp/{VERSION}", exist_ok=True)

    joblib.dump(model, f"/tmp/{VERSION}/model.joblib")
    print(f"✓ Model exported: {VERSION}")


In [ ]:
# ── Cell 9 — Push to GCS ──────────────────────────────────────────────────
if GATE_STATUS not in ("passed", "first_model") or VERSION is None:
    print(f"⏭  Cell 9 skipped — gate status is [{GATE_STATUS}]")
else:
    blob = bucket.blob(f"models/{VERSION}/model.joblib")
    blob.upload_from_filename(f"/tmp/{VERSION}/model.joblib")
    print(f"✓ Uploaded: models/{VERSION}/model.joblib")

    if WANDB_ACTIVE and run:
        try:
            wandb.log_artifact(f"/tmp/{VERSION}/model.joblib", name="model-joblib", type="model")
        except Exception as e:
            print(f"  W&B artifact log skipped: {e}")


In [ ]:
# ── Cell 10 — Update model_registry.json ─────────────────────────────────
import datetime as dt

prev_version = reg_fresh.get("latest_version")
prev_mae_val = reg_fresh.get("mae")

if GATE_STATUS in ("passed", "first_model") and VERSION is not None:
    # Full update — new model becomes production
    reg_fresh.update({
        "latest_version":   VERSION,
        "previous_version": prev_version,
        "last_trained":     dt.datetime.now(dt.timezone.utc).isoformat(),
        "model_path":       f"models/{VERSION}/model.joblib",
        "joblib_path":      f"models/{VERSION}/model.joblib",
        "mae":              mae,
        "previous_mae":     prev_mae_val,
        "rows_trained_on":  len(df),
        "feature_version":  reg_fresh["feature_version"],
        "gate_status":      GATE_STATUS,
    })
else:
    # Partial update — record that training ran but prod model unchanged
    reg_fresh.update({
        "last_trained":     dt.datetime.now(dt.timezone.utc).isoformat(),
        "gate_status":      GATE_STATUS,
        "rows_trained_on":  len(df),
        # latest_version, mae, model_path — all unchanged
    })

reg_blob = bucket.blob("model_registry.json")
reg_blob.upload_from_string(json.dumps(reg_fresh, indent=2), content_type="application/json")
reg_blob.cache_control = "no-cache, no-store, max-age=0"
reg_blob.patch()

# Always write last_training_result.json for TRAIN button UI feedback
result = {
    "status":     GATE_STATUS,
    "mae":        mae,
    "prod_mae":   prod_mae,
    "version":    VERSION,
    "trained_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    "rows":       len(df),
}
result_blob = bucket.blob("registry/last_training_result.json")
result_blob.upload_from_string(json.dumps(result, indent=2), content_type="application/json")
result_blob.cache_control = "no-cache, no-store, max-age=0"
result_blob.patch()

if WANDB_ACTIVE and run:
    wandb.finish()

# ── Final summary ─────────────────────────────────────────────────────────
print()
print("── Training Summary ─────────────────────────────────────────")
print(f"  Gate status  : {GATE_STATUS}")
print(f"  New MAE      : {mae:.4f}")
print(f"  Prod MAE     : {prod_mae:.4f}")
print(f"  Version      : {VERSION}")
print(f"  Rows trained : {len(df):,}")
print()
if GATE_STATUS in ("passed", "first_model"):
    print("🚀 Deploy function will trigger automatically (~5 min).")
    print("   Watch: cloud.sh → option 10 → deploy-model")
    print("   TRAIN button will flash green when deployed.")
elif GATE_STATUS == "marginal":
    print("⚠️  Model slightly better but below 2% threshold.")
    print("   Prod model unchanged. Generate more data and retrain.")
    print("   TRAIN button will flash yellow.")
else:
    print("❌ Model did not improve. Prod model unchanged.")
    print("   Next steps:")
    print("   1. cloud.sh → option 8 (refresh dbt + features)")
    print("   2. Generate more data")
    print("   3. Rerun this notebook")
    print("   TRAIN button will flash yellow.")
print("─────────────────────────────────────────────────────────────")


In [ ]:
# ── Cell 11 — Verify predictions ─────────────────────────────────────────
if GATE_STATUS not in ("passed", "first_model") or VERSION is None:
    print(f"⏭  Cell 11 skipped — no model exported (gate: {GATE_STATUS})")
else:
    skl_pred = model.predict(X_val[:5])
    print("Predictions (first 5 rows):")
    for i in range(5):
        print(f"  leak_row={skl_pred[i][0]:.1f}  leak_col={skl_pred[i][1]:.1f}")
    print("Verification complete.")
